In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm

In [2]:
repo_dir = os.getcwd()
repo_dir = os.path.dirname(repo_dir)

## Load Data ##

In [3]:
Metrics = xr.open_dataset(repo_dir+'/input/Metrics.nc').drop_sel(pft = 'c3_crop')
growing_season_mask = xr.open_dataset(repo_dir+'/utils/GrowingSeasonMask.nc').mask.drop_sel(pft = 'c3_crop')
prim_pft=xr.open_dataset(repo_dir+'/utils/primaryPFT.nc')

## Calculate Growing Season GPP, T, and Cveg ##

In [4]:
B_growing_season = Metrics.BTRANMN.where(growing_season_mask)
B_growing_season_annual_mean = B_growing_season.groupby("time.year").mean(dim="time")

TOTVEGC_growing_season = Metrics.TOTVEGC.where(growing_season_mask)
TOTVEGC_growing_season_annual_mean = TOTVEGC_growing_season.groupby("time.year").mean(dim="time")

GPP_growing_season = Metrics.GPP.where(growing_season_mask)
GPP_growing_season_annual_mean = GPP_growing_season.groupby("time.year").mean(dim="time")

T_growing_season = Metrics.FCTR.where(growing_season_mask)
T_growing_season_annual_mean = T_growing_season.groupby("time.year").mean(dim="time")

B_growing_season_GC = (B_growing_season* TOTVEGC_growing_season).sum(dim='pft') / TOTVEGC_growing_season.sum(dim='pft')
T_growing_season_GC = T_growing_season.sum(dim = 'pft')
GPP_growing_season_GC = GPP_growing_season.sum(dim = 'pft')

B_growing_season_annual_mean_GC = (B_growing_season_annual_mean* TOTVEGC_growing_season_annual_mean).sum(dim='pft') / TOTVEGC_growing_season_annual_mean.sum(dim='pft')
GPP_growing_season_annual_mean_GC = GPP_growing_season_annual_mean.sum(dim = 'pft')
T_growing_season_annual_mean_GC = T_growing_season_annual_mean.sum(dim = 'pft')

## Version 1 (same as main text): GPP Drought Sensitivity - Min B Years ##

### ORG Level ###

In [5]:
# Retrieve the coordinate for 'ens' (used for the fallback nan array)
ens_coord = GPP_growing_season_annual_mean.coords['ens']
ens_size = ens_coord.size

# List to collect our individual DataArrays
results = []

# Loop through every gridcell and pft with a progress bar on gridcell
for gi in tqdm(GPP_growing_season_annual_mean.coords['gridcell'].values, desc="Processing gridcells"):
    for pf in GPP_growing_season_annual_mean.coords['pft'].values:
        try:
            sub = B_growing_season_annual_mean.sel(gridcell = gi, pft = pf)
            minB_year1 = sub.mean(dim = 'ens').idxmin(dim = 'year').values.item()
            minB_year2 = sub.mean(dim = 'ens').drop_sel(year = minB_year1).idxmin(dim = 'year').values.item()
            minB_year3 = sub.mean(dim = 'ens').drop_sel(year = [minB_year1, minB_year2]).idxmin(dim = 'year').values.item()
            # Try to select the GPP slice for the given year.
            data_slice = GPP_growing_season_annual_mean.sel(gridcell=gi, pft=pf).sel(year=[minB_year1,minB_year2,minB_year3]).mean(dim = 'year')
        except Exception as e:
            # If an error occurs (e.g., year not found), assign a fallback DataArray of NaNs.
            data_slice = xr.DataArray(np.full((ens_size,), np.nan), dims=['ens'], coords={'ens': ens_coord}).rename('GPP')
        # Expand dimensions to include gridcell and pft in the resulting DataArray.
        data_slice = data_slice.expand_dims({'gridcell': [gi], 'pft': [pf]})
        results.append(data_slice)

# Combine the individual slices along the new 'gridcell' and 'pft' dimensions.
minyear_GPP_all = xr.combine_by_coords(results)
GPPDroughtSens3Years_PFT = (minyear_GPP_all - GPP_growing_season_annual_mean.mean(dim = 'year')) / GPP_growing_season_annual_mean.mean(dim = 'year')

Processing gridcells: 100%|██████████| 400/400 [00:50<00:00,  7.89it/s]


### ECO Level ###

In [6]:
# Retrieve the coordinate for 'ens' (used for the fallback nan array)
ens_coord = GPP_growing_season_annual_mean_GC.coords['ens']
ens_size = ens_coord.size

# List to collect our individual DataArrays
results = []

# Loop through every gridcell and pft with a progress bar on gridcell
for gi in tqdm(GPP_growing_season_annual_mean_GC.coords['gridcell'].values, desc="Processing gridcells"):
    try:
        sub = B_growing_season_annual_mean_GC.sel(gridcell = gi)
        minB_year1 = sub.mean(dim = 'ens').idxmin(dim = 'year').values.item()
        minB_year2 = sub.mean(dim = 'ens').drop_sel(year = minB_year1).idxmin(dim = 'year').values.item()
        minB_year3 = sub.mean(dim = 'ens').drop_sel(year = [minB_year1, minB_year2]).idxmin(dim = 'year').values.item()
        # Try to select the GPP slice for the given year.
        data_slice = GPP_growing_season_annual_mean_GC.sel(gridcell=gi).sel(year=[minB_year1,minB_year2,minB_year3]).mean(dim = 'year')
    except Exception as e:
        # If an error occurs (e.g., year not found), assign a fallback DataArray of NaNs.
        data_slice = xr.DataArray(np.full((ens_size,), np.nan), dims=['ens'], coords={'ens': ens_coord}).rename('GPP')
    
    # Expand dimensions to include gridcell and pft in the resulting DataArray.
    data_slice = data_slice.expand_dims({'gridcell': [gi]})
    results.append(data_slice)

# Combine the individual slices along the new 'gridcell' and 'pft' dimensions.
minyear_GPP_all_GC = xr.combine_by_coords(results)
GPPDroughtSens3Years_GC = (minyear_GPP_all_GC - GPP_growing_season_annual_mean_GC.mean(dim = 'year')) / GPP_growing_season_annual_mean_GC.mean(dim = 'year')


Processing gridcells: 100%|██████████| 400/400 [00:01<00:00, 310.72it/s]


## Version 2: T Drought Sensitivity - Min B Years ##

### ORG Level ###

In [7]:
# Retrieve the coordinate for 'ens' (used for the fallback nan array)
ens_coord = GPP_growing_season_annual_mean.coords['ens']
ens_size = ens_coord.size

# List to collect our individual DataArrays
results = []

# Loop through every gridcell and pft with a progress bar on gridcell
for gi in tqdm(T_growing_season_annual_mean.coords['gridcell'].values, desc="Processing gridcells"):
    for pf in T_growing_season_annual_mean.coords['pft'].values:
        try:
            sub = B_growing_season_annual_mean.sel(gridcell = gi, pft = pf)
            minB_year1 = sub.mean(dim = 'ens').idxmin(dim = 'year').values.item()
            minB_year2 = sub.mean(dim = 'ens').drop_sel(year = minB_year1).idxmin(dim = 'year').values.item()
            minB_year3 = sub.mean(dim = 'ens').drop_sel(year = [minB_year1, minB_year2]).idxmin(dim = 'year').values.item()
            # Try to select the T slice for the given year.
            data_slice = T_growing_season_annual_mean.sel(gridcell=gi, pft=pf).sel(year=[minB_year1,minB_year2,minB_year3]).mean(dim = 'year')
        except Exception as e:
            # If an error occurs (e.g., year not found), assign a fallback DataArray of NaNs.
            data_slice = xr.DataArray(np.full((ens_size,), np.nan), dims=['ens'], coords={'ens': ens_coord}).rename('FCTR')
        # Expand dimensions to include gridcell and pft in the resulting DataArray.
        data_slice = data_slice.expand_dims({'gridcell': [gi], 'pft': [pf]})
        results.append(data_slice)

# Combine the individual slices along the new 'gridcell' and 'pft' dimensions.
minyear_T_all = xr.combine_by_coords(results)
TDroughtSens3Years_PFT = (minyear_T_all - T_growing_season_annual_mean.mean(dim = 'year')) / T_growing_season_annual_mean.mean(dim = 'year')

Processing gridcells: 100%|██████████| 400/400 [00:12<00:00, 31.54it/s]


### ECO Level ###

In [8]:
# Retrieve the coordinate for 'ens' (used for the fallback nan array)
ens_coord = T_growing_season_annual_mean_GC.coords['ens']
ens_size = ens_coord.size

# List to collect our individual DataArrays
results = []

# Loop through every gridcell and pft with a progress bar on gridcell
for gi in tqdm(T_growing_season_annual_mean_GC.coords['gridcell'].values, desc="Processing gridcells"):
    try:
        sub = B_growing_season_annual_mean_GC.sel(gridcell = gi)
        minB_year1 = sub.mean(dim = 'ens').idxmin(dim = 'year').values.item()
        minB_year2 = sub.mean(dim = 'ens').drop_sel(year = minB_year1).idxmin(dim = 'year').values.item()
        minB_year3 = sub.mean(dim = 'ens').drop_sel(year = [minB_year1, minB_year2]).idxmin(dim = 'year').values.item()
        # Try to select the T slice for the given year.
        data_slice = T_growing_season_annual_mean_GC.sel(gridcell=gi).sel(year=[minB_year1,minB_year2,minB_year3]).mean(dim = 'year')
    except Exception as e:
        # If an error occurs (e.g., year not found), assign a fallback DataArray of NaNs.
        data_slice = xr.DataArray(np.full((ens_size,), np.nan), dims=['ens'], coords={'ens': ens_coord}).rename('FCTR')
    
    # Expand dimensions to include gridcell and pft in the resulting DataArray.
    data_slice = data_slice.expand_dims({'gridcell': [gi]})
    results.append(data_slice)

# Combine the individual slices along the new 'gridcell' and 'pft' dimensions.
minyear_T_all_GC = xr.combine_by_coords(results)
TDroughtSens3Years_GC = (minyear_T_all_GC - T_growing_season_annual_mean_GC.mean(dim = 'year')) / T_growing_season_annual_mean_GC.mean(dim = 'year')

Processing gridcells: 100%|██████████| 400/400 [00:05<00:00, 78.07it/s]


## Version 3: GPP Drought Sensitivity - Slope of GPP ~ B Relationship ##

In [9]:
def slope_regression(B, GPP):
    """
    Slope of y ~ a + b*x where y is the relative GPP anomaly.
    Returns NaN if not enough data or x has (near) zero variance.
    """
    with np.errstate(divide='ignore', invalid='ignore'):
        GPP_mean = np.nanmean(GPP)
        if not np.isfinite(GPP_mean) or GPP_mean == 0:
            return np.nan
        y = (GPP - GPP_mean) / GPP_mean

    mask = np.isfinite(B) & np.isfinite(y)
    if mask.sum() < 2:
        return np.nan

    x = B[mask].astype(float)
    y = y[mask].astype(float)

    x_mean = x.mean()
    y_mean = y.mean()
    den = np.dot(x - x_mean, x - x_mean)  # var * (n-1)
    if not np.isfinite(den) or den <= 1e-12:
        return np.nan

    num = np.dot(x - x_mean, y - y_mean)  # cov * (n-1)
    return num / den

### ORG Level ###

In [10]:
slope_GPP_PFT = xr.apply_ufunc(
    slope_regression,
    B_growing_season,
    GPP_growing_season,
    input_core_dims=[['time'], ['time']],
    vectorize=True,
    dask='parallelized',
    output_dtypes=[float],
)

/glade/derecho/scratch/emargiotta/tmp/ipykernel_24117/3787045856.py:7: RuntimeWarning: Mean of empty slice
  GPP_mean = np.nanmean(GPP)


### ECO Level ###

In [11]:
slope_GPP_GC = xr.apply_ufunc(
    slope_regression,
    B_growing_season_GC,
    GPP_growing_season_GC,
    input_core_dims=[['time'], ['time']],
    vectorize=True,
    dask='parallelized',
    output_dtypes=[float],
)

## Version 4: T Drought Sensitivity - Slope of T ~ B Relationship ##

In [12]:
def slope_regression(B, T):
    """
    Slope of y ~ a + b*x where y is the relative GPP anomaly.
    Returns NaN if not enough data or x has (near) zero variance.
    """
    with np.errstate(divide='ignore', invalid='ignore'):
        T_mean = np.nanmean(T)
        if not np.isfinite(T_mean) or T_mean == 0:
            return np.nan
        y = (T - T_mean) / T_mean

    mask = np.isfinite(B) & np.isfinite(y)
    if mask.sum() < 2:
        return np.nan

    x = B[mask].astype(float)
    y = y[mask].astype(float)

    x_mean = x.mean()
    y_mean = y.mean()
    den = np.dot(x - x_mean, x - x_mean)  # var * (n-1)
    if not np.isfinite(den) or den <= 1e-12:
        return np.nan

    num = np.dot(x - x_mean, y - y_mean)  # cov * (n-1)
    return num / den

### ORG Level ###

In [13]:
slope_T_PFT = xr.apply_ufunc(
    slope_regression,
    B_growing_season,
    T_growing_season,
    input_core_dims=[['time'], ['time']],
    vectorize=True,
    dask='parallelized',
    output_dtypes=[float],
)

/glade/derecho/scratch/emargiotta/tmp/ipykernel_24117/2502253491.py:7: RuntimeWarning: Mean of empty slice
  T_mean = np.nanmean(T)


### ECO Level ###

In [14]:
slope_T_GC = xr.apply_ufunc(
    slope_regression,
    B_growing_season_GC,
    T_growing_season_GC,
    input_core_dims=[['time'], ['time']],
    vectorize=True,
    dask='parallelized',
    output_dtypes=[float],
)

## Final postprocessing ##

In [15]:
DroughtSensVersions_Supp_PFT = xr.merge([
    GPPDroughtSens3Years_PFT.GPP.rename('GPPMinYears'),
    TDroughtSens3Years_PFT.FCTR.rename('TMinYears'),
    slope_GPP_PFT.rename('GPPSlope'),
    slope_T_PFT.rename('TSlope')
])

In [16]:
DroughtSensVersions_Supp_GC = xr.merge([
    GPPDroughtSens3Years_GC.GPP.rename('GPPMinYears'),
    TDroughtSens3Years_GC.FCTR.rename('TMinYears'),
    slope_GPP_GC.rename('GPPSlope'),
    slope_T_GC.rename('TSlope')
])

In [17]:
os.remove(repo_dir+'/supplement/input/DroughtSensVersions_Supp_PFT.nc')
os.remove(repo_dir+'/supplement/input/DroughtSensVersions_Supp_GC.nc')

In [18]:
DroughtSensVersions_Supp_PFT.to_netcdf(repo_dir+'/supplement/input/DroughtSensVersions_Supp_PFT.nc')
DroughtSensVersions_Supp_GC.to_netcdf(repo_dir+'/supplement/input/DroughtSensVersions_Supp_GC.nc')